# 03.1 — Built-in Tools Tour

- LocalShellBackend workspace; guided prompts covering every harness tool.
- Run top-to-bottom; each section builds on the last.
- Run: python 03.1-builtin_tools.py


## How to use

- Run cells in order; outputs show tool calls and state changes.
- Markdown before each cell names the step and the key API.
- Experiment after a clean run; re-run from the top if you edit.
- Code cells are identical to the companion `.py` script.


### Episode brief

- 03 - Built-in Tools.
- Run this cell, then check the output before moving on.
  03 - Built-in Tools
  ===================
  Deep Agents supplies filesystem tools and a `task` subagent tool automatically.
  `LocalShellBackend` also exposes `execute`; TodoListMiddleware opts into
  `write_todos`. This demo makes every tool act in a disposable lesson workspace.
  Run: python 03-builtin_tools-cx.py
  Override: DEEPAGENTS_MODEL=openai:gpt-4.1-mini python 03-builtin_tools-cx.py


### Step 1: Imports and model configuration

- Key: `from deepagents import create_deep_agent`.
- Note: Environment & Model Configuration
- Run this cell, then check the output before moving on.


In [ ]:
# %% Step 1: Imports and model configuration
import os
import time
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from dotenv import load_dotenv
from langchain.agents.middleware import TodoListMiddleware
from rich import print

############################ Environment & Model Configuration ###########################

# Pull API keys and model overrides from a local .env file
load_dotenv()

# Allow the model to be swapped without editing code; fall back to a small local model
# Override with: DEEPAGENTS_MODEL=openai:gpt-4.1-mini  (requires valid OPENAI_API_KEY)
# Default uses local Ollama so no OpenAI key is needed.
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")

# Sandbox directory that every tool call is confined to
# NOTE: __file__ is not defined in notebooks, so fall back to cwd (works in both .py and .ipynb)
try:
    _HERE = Path(__file__).resolve().parent
except NameError:
    _HERE = Path.cwd()
WORKSPACE = _HERE / "workspace"

# Checklist used at the end to verify which built-in tools the agent actually exercised
BUILT_INS = (
    "write_todos",
    "ls",
    "read_file",
    "write_file",
    "edit_file",
    "glob",
    "grep",
    "execute",
    "task",
    "delete",
)
# print(MODEL)


############################### Workspace Seeding ###############################

### Step 2: Create safe, repeatable data for the agent to explore

- Key: `def seed_workspace() -> None:`.
- Note: Recreate the data folder so reruns always start from a known state
- Run this cell, then check the output before moving on.


In [2]:
# %% Step 2: Create safe, repeatable data for the agent to explore
def seed_workspace() -> None:
    """Reset only this tutorial's sample files before each run."""
    # Recreate the data folder so reruns always start from a known state
    data = WORKSPACE / "data"
    data.mkdir(parents=True, exist_ok=True)
    # Create initial sample files for the agent to interact with
    (data / "fruits.txt").write_text("apple\nbanana\ncherry\n")
    (data / "colors.txt").write_text("red\nblue\ngreen\n")
    # Create a notes file for the agent to edit
    (WORKSPACE / "notes.md").write_text("# Notes\n- status: draft\n")
    # Ensure the veggies file is initially absent so the agent can create it
    (data / "veggies.txt").unlink(missing_ok=True)


################################ Agent Construction ################################

### Step 3: Build an agent with every available harness tool

- Key: `create_deep_agent(backend, model, system_prompt)`.
- Note: Jail shell + filesystem access to the workspace and cap runaway commands
- Run this cell, then check the output before moving on.


In [3]:
# %% Step 3: Build an agent with every available harness tool
def build_agent():
    """LocalShellBackend adds execute; middleware adds write_todos."""
    # Jail shell + filesystem access to the workspace and cap runaway commands
    backend = LocalShellBackend(root_dir=str(WORKSPACE.resolve()), timeout=30)
    return create_deep_agent(
        model=MODEL,
        backend=backend,
        middleware=[TodoListMiddleware()],  # Opt in to the write_todos planning tool
        # Force explicit tool usage so the demo can prove each tool works
        system_prompt="""You are demonstrating Deep Agents tools in a throwaway workspace.
            For every requested operation, call the named tool exactly; never replace a filesystem
            operation with execute.Use relative paths. Keep final answers brief. workspace
            path is: ./ all file operations should be relative
            to this path.""",
    )


############################ Prompt Script & Tool Tracking ###########################

### Step 4: Short prompts that explicitly cover every tool

- Key: `def tool_names(messages: list) -> set[str]:`.
- Note: Each prompt names the tools it should trigger, spreading coverage across turns
- Run this cell, then check the output before moving on.


In [4]:
# %% Step 4: Short prompts that explicitly cover every tool
# Each prompt names the tools it should trigger, spreading coverage across turns
PROMPTS = [
    (  # tools: ls, read_file, write_todos
        "1. Use write_todos to plan these two items, then use ls on data and "
        "read_file on data/fruits.txt and tell me what fruits are listed."
    ),
    (  # tools: edit_file, write_file
        "2. Use write_file to create data/veggies.txt containing carrot, pepper, and basil on "
        "separate lines. Then use edit_file to change 'draft' to 'final' in notes.md."
    ),
    (  # tools: glob, grep, execute
        "3. Use glob to find every .txt file. Use grep to find the file containing 'blue'. "
        "Use execute to run exactly: wc -l data/*.txt"
    ),
    (  # tools: task, delete, write_todos
        "4. Use task to delegate counting all lines in data/*.txt to a subagent."
        "Then use the delete tool to remove data/veggies.txt, complete the "
        "todos, and summarize the results."
    ),  # delete
    ("5. use delete tool to remove a file from the workspace and tell me the file you removed"),
    (
        "6. List all built-in tools available to you in deepagents with a brief description of "
        "each, and write each tool's name and description to a markdown file named tools.md  "
        "or edit_file tool. in the workspace directory using the write_file"
    ),
]


def tool_names(messages: list) -> set[str]:
    """Collect tool names from the agent's LangChain messages."""
    # Flatten tool_calls across all messages; plain text messages simply contribute nothing
    tool_names = set()
    # Iterate over all messages and collect the names of tools that were called
    for message in messages:
        # Skip messages with no tool calls (i.e., "tool_calls" attribute is missing or empty)
        for call in getattr(message, "tool_calls", []) or []:
            tool_names.add(call["name"])
    # Return the collected set of tool names
    return tool_names


############################ Conversation Loop & Coverage Report ###########################

### Step 5: Run the guided conversation and report actual coverage

- Key: `def main() -> None:`.
- Note: Initialize the agent, message history, and set of called tools
- Run this cell, then check the output before moving on.


In [5]:
# %% Step 5: Run the guided conversation and report actual coverage
def main() -> None:
    """Invoke each prompt, preserving messages so the agent can continue its plan."""
    start = time.time()  # Track wall-clock cost of the whole multi-turn run
    seed_workspace()
    # Initialize the agent, message history, and set of called tools
    agent, messages, called = build_agent(), [], set()
    print(f"[dim]model: {MODEL}  workspace: {WORKSPACE}[/dim]")

    # Replay the full history each turn so the agent keeps its todo list context
    for turn, prompt in enumerate(PROMPTS, start=1):
        print(f"\n[bold magenta]--- Turn {turn} ---[/bold magenta]")
        print(f"Prompt: {prompt}")
        # Invoke the agent with the current prompt and the accumulated message history
        result = agent.invoke({"messages": messages + [{"role": "user", "content": prompt}]})
        # print("Got results finding new tool calls...")
        # Update the message history and determine which tools were newly called this turn
        messages = result["messages"]
        tools_used = tool_names(messages)
        # Determine which tools were newly called this turn
        new_calls = tools_used - called  # Report only tools first seen this turn

        called.update(new_calls)  # Accumulate coverage across every turn
        print("\nTools Used:", ", ".join(sorted(tools_used)) or "none")
        print("\nAnswer: \n")
        # print(messages[-1].content)  # Show only the agent's latest reply
        print(messages[-1])

    # Score the run against the expected built-in tool list
    print("\n[bold cyan]Built-in tool coverage[/bold cyan]")
    for name in BUILT_INS:
        # Check if the current built-in tool was called during the conversation
        marker = "[green]yes[/green]" if name in called else "[red]no[/red]"
        print(f"  {name:11} {marker}")  # Pad names so the yes/no column lines up
    # print(f"\n[bold green]Final answer:[/bold green] {messages[-1].text}")
    end = time.time()
    print(f"\n[bold yellow]Total execution time: {end - start:.2f} seconds[/bold yellow]")


if __name__ == "__main__":
    main()

model: metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15', 'langchain-openai': '1.5.1'}} 
profile={'name': 'GPT-4.1 nano', 'status': 'deprecated', 'release_date': '2025-04-14', 'last_updated': 
'2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 
'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 
'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 
'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 
'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} 
client=<openai.resources.chat.completions.completions.Completions object at 0x137b267b0> 
async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x137b27230> 
root_client=<openai.OpenAI object at 0x136f7c980> root_async_client=<openai.AsyncOpenAI object at 0x137b26f90> 
model_name='gpt-4.1-nano' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True 
stream_chunk_timeout=120.0  workspace: 
/Users/james/Library/CloudStorage/Dropbox/GitHub/YouTube/LangChain_DeepAgents/deepagents_101/03.1-builtin_tools/wor
kspace

--- Turn 1 ---

Prompt: 1. Use write_todos to plan these two items, then use ls on data and read_file on data/fruits.txt and tell 
me what fruits are listed.

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: "sk-proj********************************************************************************************************************************************************************************************************************enai. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}